In [1]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Some global variables

In [2]:
filter_start = "2025-10-01"
filter_end = "2026-10-01"

period_start = "2023-10-01"
period_end = "2026-10-01"

In [3]:
# Set up exchange rate dataframe for every day in the period

exchange_rate = yf.Ticker("CAD=X").history(start = period_start, end = period_end)["Close"]

# Keep only year month date, datetime objects, in index column of dataframe
exchange_rate.index = pd.to_datetime(exchange_rate.index.strftime('%Y-%m-%d'))

display(exchange_rate.head())

Date
2023-10-02    1.35737
2023-10-03    1.36763
2023-10-04    1.37091
2023-10-05    1.37442
2023-10-06    1.37057
Name: Close, dtype: float64

Extracting stocks

In [4]:
# EXTRACTING TICKERS. MAKE SURE TO CHANGE FILE NAME FOR ACTUAL SUBMISSION
tickers = pd.read_csv("Tickers.csv", header=None)[0].to_list()
close_prices = pd.DataFrame()

for tick in tickers:
    t = yf.Ticker(tick)
    information = t.info

    # Check if there is data in here (i.e. not a delisted or invalid ticker)
    if t.history(period='max').empty:
        print(tick, "doesn't exist")
        continue
    
    # Check if it's a CAD or USD stock
    if (information["market"] != "ca_market") and (information["market"] != "us_market"):
        print(tick, "wrong market")
        continue

    # Check if it's a stock and not something like a bond or index fund
    if information["quoteType"] != "EQUITY":
        print(tick, "not stock")
        continue

    if (information["currency"] != "CAD") and (information["currency"] != "USD"):
        print(tick, "currency wrong")
        continue
    

    temp = yf.download(tick, filter_start, filter_end)

    if temp["Volume"].mean()[tick] < 5000:
        print(tick, "not enough volume")
        continue
    
    close_prices[tick] = temp["Close"]

# Filters out the first and last trading day of each month
idx = close_prices.index.to_series()
date_series_first=idx.groupby([idx.dt.year, idx.dt.month]).first()
date_series_last=idx.groupby([idx.dt.year, idx.dt.month]).last()
monthly_first = []
monthly_last = []
for i in date_series_first:
    monthly_first.append(i)
for i in date_series_last:
    monthly_last.append(i)

for i in range(len(monthly_first)):
    start_ind = close_prices.index.get_loc(monthly_first[i])
    end_ind = close_prices.index.get_loc(monthly_last[i])

    if (end_ind - start_ind + 1) < 18:
        close_prices = close_prices.drop(close_prices.loc[monthly_first[i]:monthly_last[i]].index)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$AGN: possibly delisted; no timezone found


AGN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$BK: possibly delisted; no price data found  (1d 1927-12-21 -> 2026-11-26)


BK doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$CELG: possibly delisted; no timezone found


CELG doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$MON: possibly delisted; no timezone found


MON doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$RTN: possibly delisted; no timezone found


RTN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [5]:
display(close_prices.head())

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,254.316589,235.580231,129.656540,236.889038,77.725349,220.630005,324.080383,215.199997,49.622253,6.67,...,65.924820,163.093613,208.619995,20.261114,107.826416,174.982285,339.266205,229.894775,78.023788,46.106377
2025-10-02,255.989151,228.041824,129.190247,237.501389,77.725349,222.410004,325.974274,217.429993,49.426430,6.63,...,68.169777,165.405457,211.240005,19.976391,106.874413,176.854431,344.545593,229.679291,79.124153,46.135265
2025-10-03,256.875153,225.487244,130.744553,238.453979,79.072174,219.509995,325.737549,216.300003,49.583088,6.28,...,68.486252,165.728699,224.889999,20.104975,108.913040,174.914383,350.857544,231.912247,80.243004,46.289333
2025-10-06,255.551102,221.901184,129.918839,241.224197,81.024086,220.899994,327.463654,219.729996,49.338310,6.68,...,70.503754,165.180145,229.520004,19.893730,108.951508,176.359711,349.464630,232.509674,80.095055,45.952305
2025-10-07,255.342056,224.446136,129.219406,244.198547,82.029327,221.779999,323.498505,221.820007,49.240395,6.48,...,73.787132,162.084625,225.100006,19.875362,108.605324,171.742401,354.227814,227.074219,79.503273,45.740448


In [6]:
full_data = pd.DataFrame()

cols = {}
for tick in close_prices:
    prices = yf.download(tick, period_start, period_end)["Close"][tick]
    if yf.Ticker(tick).info["currency"] == "USD":
        prices = prices * exchange_rate.reindex(prices.index).ffill()
    cols[tick] = prices
    
full_data = pd.concat(cols, axis=1, sort=True).sort_index()

full_data["CBIL.TO"] = yf.download("CBIL.TO", period_start, period_end, auto_adjust=True)["Close"]

full_data = full_data.ffill()

display(full_data.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,CBIL.TO
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,232.519125,180.319221,121.325726,393.012968,76.241152,175.725132,195.350702,254.954813,33.728010,6.02,...,141.522611,73.839996,17.100687,70.040115,198.282192,655.916130,257.465521,174.720008,38.394417,45.243145
2023-10-03,232.456387,180.677246,122.319412,393.376058,75.013726,170.570816,190.994134,258.372658,32.977460,5.98,...,141.424401,70.440002,17.204720,69.378128,196.408601,654.323978,260.871020,174.530914,37.721729,45.229580
2023-10-04,234.716907,181.429975,122.433547,398.388435,76.116891,174.105576,193.537624,255.990027,33.094827,5.86,...,141.326150,72.230003,17.756865,69.195206,197.781136,657.373793,259.558962,174.915308,37.691547,45.243145
2023-10-05,237.011686,181.598940,123.452841,399.757000,77.327376,173.121948,195.441625,256.040701,33.333050,5.20,...,141.623504,71.510002,18.100962,69.997551,194.697581,666.297682,258.126512,174.975274,38.550098,45.247665
2023-10-06,239.833972,182.060525,123.977218,401.894164,78.033789,175.378129,197.848890,256.817403,33.252432,5.15,...,142.615890,73.269997,18.044945,70.359039,196.517909,675.474405,259.738334,175.497638,38.430051,45.256702


Sharpe calculation

In [7]:
half_life = 250

In [8]:
def get_stats(tick, half_life):
    s = full_data[tick].dropna()
    delta = np.log(1 + s.pct_change()).ewm(halflife = half_life)
    standard_deviation = delta.std().iloc[-1]
    average = delta.mean().iloc[-1]

    return average, standard_deviation

In [9]:
Rf = get_stats("CBIL.TO", half_life)[0]
print("Risk free return:", Rf)

sharpe = {}

for tick in close_prices:
    stats = get_stats(tick, half_life)
    sharpe[tick] = (stats[0] - Rf)/stats[1]

sharpe = dict(sorted(sharpe.items(), key = lambda item:item[1]))
print(sharpe)

Risk free return: 0.00010671468228646669
{'T.TO': np.float64(-0.09193762876563893), 'ACN': np.float64(-0.03509386489473794), 'PEP': np.float64(-0.03230759544264417), 'UPS': np.float64(-0.019860281674475292), 'ABT': np.float64(-0.01778745801740057), 'PYPL': np.float64(-0.014007785402048509), 'BA': np.float64(-0.013166570450088786), 'UNH': np.float64(-0.00882485271425503), 'CL': np.float64(0.003153015900331438), 'AIG': np.float64(0.0036821159959596972), 'LMT': np.float64(0.008114174099371122), 'AXP': np.float64(0.016000908337639605), 'QCOM': np.float64(0.018020356663380147), 'BLK': np.float64(0.02292016015806593), 'AMZN': np.float64(0.026520475276444418), 'PFE': np.float64(0.027053168792155365), 'BIIB': np.float64(0.028587262689162205), 'SHOP.TO': np.float64(0.02931816933523343), 'UNP': np.float64(0.030630439797287762), 'BMY': np.float64(0.03198046646807654), 'LLY': np.float64(0.0397052443127103), 'BAC': np.float64(0.040769463534430936), 'TXN': np.float64(0.04130310520497358), 'USB': np.

In [10]:
def weighted_corr(t1, t2):
    d1 = full_data[t1].pct_change(fill_method=None)
    d2 = full_data[t2].pct_change(fill_method=None)
    return d1.ewm(halflife=half_life).corr(d2).dropna().iloc[-1]

Correlation matrix

In [11]:
corr_mat = pd.DataFrame()

for t1 in close_prices:
    for t2 in close_prices:
        corr_mat.loc[t1, t2] = weighted_corr(t1, t2)

In [12]:
display(corr_mat)

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
AAPL,1.000000,0.194254,0.132854,0.215326,0.277563,0.279194,0.355878,0.312301,0.340458,0.113994,...,0.312324,0.317230,0.338428,0.194442,0.211206,0.277676,-0.001601,0.246692,0.300750,0.359230
ABBV,0.194254,1.000000,0.337372,0.186326,0.253049,-0.053362,0.157285,0.060093,0.162880,-0.014198,...,0.144141,0.010036,0.028752,0.074433,0.053394,0.071150,0.139082,0.254595,0.226953,0.205309
ABT,0.132854,0.337372,1.000000,0.267511,0.246025,-0.007405,0.146430,0.115855,0.176014,-0.145556,...,0.168768,-0.041628,0.116199,0.089898,0.034980,-0.012994,0.106915,0.291408,0.181668,0.212030
ACN,0.215326,0.186326,0.267511,1.000000,0.225876,0.162441,0.322218,0.139568,0.185739,0.062243,...,0.344292,0.069002,0.365620,0.222115,0.024306,-0.085630,0.110766,0.165150,0.174303,0.272432
AIG,0.277563,0.253049,0.246025,0.225876,1.000000,0.114573,0.318293,0.213833,0.354936,-0.012067,...,0.174231,0.093020,0.149236,0.025890,0.170660,0.132204,0.123403,0.352019,0.191967,0.359135
AMZN,0.279194,-0.053362,-0.007405,0.162441,0.114573,1.000000,0.401539,0.331861,0.316599,0.191711,...,0.302479,0.281235,0.344012,-0.197243,0.209295,0.243408,0.050052,0.149051,0.197262,0.313900
AXP,0.355878,0.157285,0.146430,0.322218,0.318293,0.401539,1.000000,0.411301,0.667997,0.193264,...,0.387069,0.308753,0.412548,0.036176,0.394655,0.289367,0.178149,0.333913,0.333174,0.700108
BA,0.312301,0.060093,0.115855,0.139568,0.213833,0.331861,0.411301,1.000000,0.360954,0.118260,...,0.269480,0.317297,0.232243,-0.000760,0.251275,0.321524,0.110067,0.187327,0.271724,0.346612
BAC,0.340458,0.162880,0.176014,0.185739,0.354936,0.316599,0.667997,0.360954,1.000000,0.160197,...,0.271996,0.278173,0.282605,-0.022011,0.444876,0.291087,0.122073,0.349899,0.334237,0.769056
BB.TO,0.113994,-0.014198,-0.145556,0.062243,-0.012067,0.191711,0.193264,0.118260,0.160197,1.000000,...,0.146719,0.255018,0.224028,-0.046085,0.208502,0.221540,0.051965,0.092423,0.179718,0.186197


In [13]:
period = 15
threshold = 1.5

In [14]:
# Note: this shouldn't be used as a raw value, as there may be periods where the variance
# jumps are spread across two periods and thus not counted. Use a threshold lower than
# what you're truly looking for. 
def count_jumps_std(period, threshold):
    result = pd.DataFrame()
    std_dict = {}
    for tick in close_prices:
        result.loc[tick, "Occurrences"] = 0
        std_dict[tick] = full_data[tick].std()
    interval = [len(full_data)-period, len(full_data)]
    while interval[0] >= 0:
        for tick in close_prices:
            if full_data[tick].iloc[interval[0]:interval[1]].std() >= std_dict[tick] * threshold:
                result.loc[tick, "Occurrences"]+=1
        interval[0]-=period
        interval[1]-=period

    return result

In [15]:
period = 15
threshold = 0.8

In [16]:
# Note: this shouldn't be used as a raw value, as there may be periods where the price
# jumps are spread across two periods and thus not counted. Use a threshold lower than
# what you're truly looking for. 

def count_price_plummets(period, threshold):
    result = pd.DataFrame()
    std_dict = {}
    for tick in close_prices:
        result.loc[tick, "Occurrences"] = 0
    interval = [len(full_data)-period, len(full_data)-1]
    while interval[0] >= 0:
        for tick in close_prices:
            if full_data.iloc[interval[1]][tick] <= full_data.iloc[interval[0]][tick] * threshold:
                result.loc[tick, "Occurrences"]+=1
        interval[0]-=period
        interval[1]-=period

    return result

In [17]:
sectors = {}

for tick in close_prices:
    sectors[tick] = yf.Ticker(tick).info.get("sector")

High growth

In [18]:
#Momentum
def get_momentum():
    prices = full_data[close_prices.columns]
    
    return prices.asof(pd.Timestamp("2026-09-01")) / prices.asof(pd.Timestamp("2025-09-01")) - 1


In [19]:
#Dividends
def get_dividends():
    result = pd.Series()
    for tick in close_prices:
        div = yf.Ticker(tick).dividends
        div.index = div.index.tz_localize(None).normalize()

        div = div[(div.index >= period_start) & (div.index < period_end)]

        if div.empty:
            result[tick] = 0
            continue

        if yf.Ticker(tick).info.get("currency") == "USD":
            div = (div * exchange_rate.asof(div.index))

        price = full_data[tick].asof(div.index)

        yields = div/price

        age = (pd.Timestamp(filter_end) - div.index).days
        weights = 0.5**(age/half_life)

        result[tick] = np.sum(yields*weights)

    return result

In [20]:
#Positive return?
def sharpe_check():
    result = pd.Series()
    for tick in close_prices:
        result[tick] = min(0, sharpe[tick])

    return result

In [21]:
#Revenue geometric mean
def revenue_CAGR():
    result = pd.Series()
    for tick in close_prices:
        stmt = yf.Ticker(tick).income_stmt
        if "Total Revenue" not in stmt.index:
            result[tick] = np.nan
            continue

        
        rev = stmt.loc["Total Revenue"].dropna().sort_index()

        if len(rev) < 2:
            result[tick] = np.nan
            continue

        years = (rev.index[-1] - rev.index[0]).days/365.25
        result[tick] = pow((rev.iloc[-1]/rev.iloc[0]), (1/years)) - 1

    return result

In [22]:
#Revenue geometric mean
def revenue_std():
    result = pd.Series()
    for tick in close_prices:
        stmt = yf.Ticker(tick).income_stmt
        if "Total Revenue" not in stmt.index:
            result[tick] = np.nan
            continue
        
        rev_delta = stmt.loc["Total Revenue"].dropna().sort_index().pct_change().dropna()

        result[tick] = rev_delta.std()
        if len(rev_delta) <= 2:
            result[tick] = np.nan

    return result

In [23]:
def z_score(ser):
    z = (ser-ser.mean())/ser.std()
    return z.fillna(z.max())

In [24]:
score_df = pd.DataFrame()

score_df["Drops"] = count_price_plummets(15, 0.8)
score_df["Momentum"] = z_score(get_momentum())
score_df["Dividends"] = z_score(get_dividends())
score_df["Sharpe Check"] = z_score(sharpe_check())
score_df["Revenue CAGR"] = z_score(revenue_CAGR())
score_df["Revenue STD"] = z_score(revenue_std())

weights = {"Drops": -1.5, "Momentum": 1, "Dividends": 0.375, "Sharpe Check": 0.4, "Revenue CAGR": 0.875, "Revenue STD": 0.25}

overall_score = sum(score_df[x] * weights[x] for x in weights).to_dict()

In [25]:
overall_score = dict(sorted(overall_score.items(), key=lambda x: -x[1]))

min_val = min(overall_score.values())
overall_score = {key: val - min_val for key, val in overall_score.items()}

overall_score

{'LLY': 7.421902259801561,
 'TD.TO': 5.446273099878385,
 'CAT': 5.291429657623881,
 'MRK': 5.279509749916327,
 'RY.TO': 4.885797705431454,
 'USB': 4.3109295181753655,
 'BMY': 4.235815766489239,
 'AAPL': 4.045298135279996,
 'C': 3.8676967040493597,
 'PM': 3.7194965817061414,
 'BIIB': 3.634773277751226,
 'KO': 3.6295560096342907,
 'BLK': 3.610255371839077,
 'BAC': 3.588834551659601,
 'ABBV': 3.4879269669375765,
 'LMT': 3.414748705039178,
 'AMZN': 3.322253082764267,
 'MO': 3.2744605056132077,
 'QCOM': 3.2040821956213605,
 'UNP': 3.103338530800271,
 'AXP': 3.1002038741562123,
 'CL': 2.9361873357966073,
 'BA': 2.9100269060172415,
 'UPS': 2.867278877076438,
 'PFE': 2.811552410843929,
 'PEP': 1.9934427212666412,
 'BB.TO': 1.749003592641337,
 'AIG': 1.6958549240404022,
 'SHOP.TO': 1.6840463720521257,
 'TXN': 1.5204760741436294,
 'ABT': 1.4514183558224358,
 'UNH': 1.2262803339645534,
 'ACN': 1.0911462622955441,
 'T.TO': 0.5664612665529254,
 'PYPL': 0.0}

In [26]:
num_stocks = 12

In [27]:
def get_best_stocks():
    sector_chosen = {}

    chosen_stocks = []

    chosen = 0
    
    for tick, score in overall_score.items():
        sector = sectors[tick]

        if chosen >= num_stocks:
            break
        
        if (sector in sector_chosen) and len(sector_chosen[sector]) == 3:
            continue
        else:
            if sector not in sector_chosen:
                sector_chosen[sector] = [tick]
            else:
                sector_chosen[sector].append(tick)

            chosen_stocks.append(tick)
            chosen+=1

    return chosen_stocks

chosen_stocks = get_best_stocks()

In [28]:
principal = 1000000

In [29]:
def get_weights(chosen_stocks):
    total_weight = sum(overall_score[x] for x in chosen_stocks)

    total_budget = 1.0

    value = {}

    sector_weights = {}
    
    for tick in chosen_stocks:
        value[tick] = min(0.15, total_budget * overall_score[tick]/total_weight, 0.4 - sector_weights.get(sectors[tick], 0))

        sector_weights[sectors[tick]] = sector_weights.get(sectors[tick], 0) + value[tick]

        total_weight -= overall_score[tick]
        total_budget -= value[tick]

    return value

get_weights(chosen_stocks)

{'LLY': 0.13493629208899588,
 'TD.TO': 0.0990177278111016,
 'CAT': 0.09620254657848347,
 'MRK': 0.095985832845023,
 'RY.TO': 0.08882782380988555,
 'USB': 0.07837624698861748,
 'BMY': 0.07701061715644958,
 'AAPL': 0.07354684980502647,
 'PM': 0.06762350939212461,
 'KO': 0.06598831576131235,
 'LMT': 0.06208294215477944,
 'AMZN': 0.0604012956082006}

**Why does your portfolio make financial sense? Explain the characteristics you favour, how stocks are selected and weighted, and why this may help achieve the client’s objective.**

### Metric 1: Number of drops

For this metric, we calculated the number of times the stock dropped 20\% within 3 trading weeks. This metric is useful for us to consider how often there's bad news that can drastically plummet the stock. 

For each of these drops, I made it such that the drop decreases score by 1.5 units, as when we're investing in stocks, we want consistent growth. When we have to deal with a stock that has a history of plummeting in price, we cannot expect them to have a stable long-term growth pattern, and hence why we weighed each drop in stock price so harshly. 


### Metric 2: Price momentum

For this metric, we calculated the averege growth in price of the stock, as an annual growth percentage, for a time frame of as long as possible. 

This metric was also weighed heavily, as this is the key indicator that the stock is trending upwards. We chose so that we can use the performance of the stock price in the past year to correlate this to the performance of the stock in the next year. 

Our measurement window ends one month before our price data does. Over very short periods (about one month), stock returns tend to partially reverse, that is, a stock that rose sharply last month is slightly more likely to fall back this month. Including that month would add noise that works against the longer-term trend we're trying to capture, 

### Metric 3: Dividends paid

Dividends paid represents a company that has nothing better to do with their money than to pay it out to shareholders. But essentially what this means is that this is a "mature" company, that people do not expect to grow much anymore. On the other hand, a company that doesn't pay dividends is likely a company that is growing. 

However, this can't be used as a perfect indicator that a company is growing. For example, most tech firms aren't expected to pay out dividends, while most banks consistently pay out dividends. This is why we used it, but with a lower weight. 

This can actually tell us contradicting things: 
1. If a tech firm starts paying off dividends, it could mean that they're no longer expecting growth, which is why the dividends were paid out.

2. On the other hand, if a bank stopped paying dividends, it could mean that they're doing terrible, to the point of not being able to pay out dividends anymore, or they're expecting more growth.

### Metric 4: Sharpe

For this, we checked for the sharpe of each of the companies. 

Raw return alone favours stocks that rose sharply, but with an unpredictable growth. For example, a stock with an expected growth of 10x per year sounds great on paper, but if it has a volatility that is 1000x of the stock's price, the 10x year on year growth all of a sudden doesn't mean much anymore. 

We calculated the daily returns as the log of the returns, as this allows us to add them up over time, which also means that taking the arithmetic mean of the data is actually meaningful, as it tells us the average log return. 

However, we set this value to 0 for stocks with positive sharpe.

This is because we don't want to overly weigh the stocks with a low average, but even lower return on investment. We only want to penalize the stocks with negative return, lower than the risk-free rate. 

### Metric 5: Revenue CAGR

This calculates the average revenue growth rate, for as long of a period as possible. 

The higher the average revenue growth is, the more we can expect good news of the company to come out, which in turn tells us that the company's stock price should be going up. 

We chose revenue over earnings because revenue is more stable and harder to distort, as gross profit can be different based on tax changes or accounting choices. 

### Metric 6: Revenue STD

Similar to the Revenue CAGR, we want to see how the revenue grows, but also how consistently it grows. A low value means that the company grows at a steady, predictable rate each year. Because the metric uses growth rates, the company size also doesn't affect the stock. 

### Normalizing with z-score

Since a lot of the values we're getting are raw score, the values from one sector can be completely different from the values of another sector. 

The way we fixed this was to get the z-score, so the weightings for each sector could be easily compared, without having to adjust the weights to reflect the magnitude of each stock. However, the drops were a raw value, since they're discrete integers with a relatively low maximum. 

**What evidence supports your strategy? Use statistics and the equal-weight comparison below to evaluate your approach, including any meaningful trade-offs.**

The equal-weight comparison had an three-year return of 45.5\%, whereas the optimized portfolio had a return of 49.7\% over the same period. This means the weighting did help evaluate our approach. 

However, based off our data, over the same period of time, the optimized portfolio had a daily standard deviation of 1.27\%, where the equal weight portfolio had a daily standard deviation of 1.12\% over the same period of time. This tells us that the meaningful tradeoff for the extra 4.2% return is the extra risk that we're dealing with when creating the portfolio. 

And this makes sense. 

A portfolio, if we're considering the same stocks, if we increase the return we should naturally a high-return portfolio to have a higher variance, as we're putting more money into higher returning, but high volatility stocks. 

The reason why we chose revenue, dividends and price momentum for our stock prices is because we wanted to analyze the underlying causes for price increases and how prices change relative to the company's performance. However, solely relying on the background information and not considering the stock price in itself could lead to sector-wide events, which is why we tracked momentum and the number of drops, so we can see how the stock is doing in general, and if there were any really bad events that we should be aware of. 

**What could go wrong? Identify at least one important weakness or situation in which the strategy may perform poorly.**

At the moment, the portfolio isn't as diversified as it looks. Although the sector cap limits how many stocks come from each sector, the stocks within each sector are closely correlated. For a well-performing sector, we can have up to 40\% of the portfolio in this sector. Since all of these stocks in that sector are closely correlated, a crash in that sector can be catastrophic. 

Additionally, one factor that our portfolio never considers is the raw magnitude of several factors, such as revenue and stock prices. This means that a penny stock with high momentum could be chosen, whereas there was no differentiation between a company with an annual revenue of 1 million dollars and 1 billion dollars. This may be important to consider, but from our knowledge of companies, this is not enough to create a metric. 